# 02 Feature engineering (member B)

Feature steps live in `src/features.py`. Use this notebook to:
1. Check that each step produces the right output
2. Try new features
3. Run the ablation to answer "which feature engineering was most useful?"

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
from src.data import load_data
from src.features import STEPS, build_features
train, test = load_data()
print("Available steps:", list(STEPS))

## 1. Check step by step

In [ ]:
steps = []
for name in STEPS:
    steps.append(name)
    X_tr, X_te = build_features(train, test, steps)
    print(f"{' + '.join(steps):<70} -> {X_tr.shape[1]} features, NaN={X_tr.isna().sum().sum()}")

In [ ]:
# How do the new features relate to price?
X_tr, _ = build_features(train, test, ["fix_types", "fill_missing", "encode_ordinal", "add_features"])
new = ["TotalSF", "TotalBath", "HouseAge", "RemodAge", "TotalPorchSF"]
X_tr[new].corrwith(np.log1p(train[TARGET])).sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(x=X_tr["TotalSF"], y=train[TARGET])
plt.title("TotalSF vs SalePrice"); plt.tight_layout(); plt.savefig(FIGURE_DIR / "totalsf_vs_price.png", dpi=150)

## 2. Try new features

Try them here first. If one helps, move it into `src/features.py` (as a new function added to `STEPS`), then add it to iter3 in `versions.py`.

In [ ]:
# Example:
# def add_my_feature(df):
#     df = df.copy()
#     df["QualxArea"] = df["OverallQual"] * df["GrLivArea"]
#     return df

## 3. Ablation: which step helps most?

Bigger `delta` = score gets worse when removed = more important step. Also saved to `reports/ablation_log.csv`.

In [ ]:
from src.pipeline import ablation
result = ablation("iter3_features")
result

In [ ]:
r = result[result.removed_step != "(none)"]
plt.figure(figsize=(7, 3)); sns.barplot(data=r, x="delta", y="removed_step")
plt.title("CV RMSE increase when step is removed"); plt.tight_layout()
plt.savefig(FIGURE_DIR / "ablation.png", dpi=150)

## Conclusion (for the report: most useful feature engineering + why)

- 